# Saighi A_k decay sweep — seed 1, 4 decay values in parallel

Follow-up to the n=10 A_k verification (decay=0.0). Result was a **falsification of report 034's optimistic seed-1 prototype**: Δtop1 = -0.051 (1/10 seeds positive), ΔR@10 = +0.009 (CI crosses 0). Saighi p. 4 explicitly notes that monotonic A_k growth (decay=0) is too aggressive — gain-vs-decay is a trade-off knob.

This sweeps decay ∈ {0.02, 0.05, 0.10, 0.20} on seed 1 with `--inhibition-gain 0.01 --n-cues 3000`, all in parallel. ~7-10 min on H100.

**What we're watching:**
- Does Δtop1 (C-A) move toward 0 (or positive) as decay grows? That's the falsifier — if higher decay closes the gap, A_k just needs a forgiveness window. If it doesn't, A_k is wrong-shaped regardless.
- A_max telemetry: with decay, A_max should saturate at a steady state rather than grow unboundedly.

Outputs to `reports/phase34_saighi_decay{D}_seed1/`.

In [ ]:
# 1. Clone the repo.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -3

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps + sanity check (CPU only in parent — keep GPU free for workers).
!pip install -q datasets
import os, sys, gc
sys.path.insert(0, 'src')
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
cfg = ConsolidationConfig(m=4, alpha=0.25, inhibition_gain=0.01, inhibition_decay=0.05)
s = ConsolidationState(cfg, device='cpu')
s.add_pattern()
s.accumulate_inhibition(0)
s.step_dynamics()  # should apply decay
expected = 0.01 * (1 - 0.05)
assert abs(s.A[0].item() - expected) < 1e-6, f'decay not applied: A={s.A[0].item()}, expected {expected}'
print(f'decay check: A = {s.A[0].item():.6f} (expected {expected:.6f}) ✓')
del cfg, s; gc.collect()

# Pre-warm wikitext cache.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows')
del splits; gc.collect()

In [ ]:
# 4. Launch 4 parallel runs: seed 1 at decay in {0.02, 0.05, 0.10, 0.20}.
# Each uses ~1.6 GB GPU. Total ~6.4 GB on 95 GB H100 NVL.
import subprocess, os, time, signal
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
DECAYS = [0.02, 0.05, 0.10, 0.20]
log_root = Path('reports/phase34_saighi_decay_sweep_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(decay):
    tag = f'dec{int(decay*100):03d}'
    out_dir = f'reports/phase34_saighi_{tag}_seed1'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        ['python', 'experiments/19_phase34_integrated.py',
         '--updater-kind', 'hebbian',
         '--device', 'cuda',
         '--seed', '1',
         '--n-cues', '3000',
         '--checkpoint-every', '500',
         '--success-threshold', '0.3',
         '--death-threshold', '0.05',
         '--death-window', '10',
         '--inhibition-gain', '0.01',
         '--inhibition-decay', str(decay),
         '--no-reencode-discovered',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return decay, tag, proc, logf, out_dir

procs = {d: launch(d) for d in DECAYS}
print(f'launched {len(procs)} runs ({time.strftime("%H:%M:%S")}):')
for d, (_, tag, proc, _, out) in procs.items():
    print(f'  decay={d} ({tag}): pid={proc.pid} -> {out}')
print()

t0 = time.time()
remaining = dict(procs)
poll_count = 0
try:
    while remaining:
        done_this = []
        for d, (decay, tag, proc, logf, out_dir) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] decay={d}: {ok}')
                done_this.append(d)
        for d in done_this:
            del remaining[d]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                elapsed = (time.time() - t0) / 60
                print(f'  --- snapshot at {elapsed:.1f} min ---')
                gpu = subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,utilization.gpu','--format=csv,noheader'], stderr=subprocess.DEVNULL).decode().strip()
                print(f'  GPU: {gpu}')
                for d, (_, tag, proc, _, _) in remaining.items():
                    log = log_root / f'{tag}.log'
                    size = log.stat().st_size if log.exists() else 0
                    try:
                        line = subprocess.check_output(['tail','-1',str(log)], stderr=subprocess.DEVNULL).decode().strip()[:90]
                    except Exception:
                        line = ''
                    print(f'  decay={d}: {size:>7}b  {line}')
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted — killing all !!!')
    for d, (_, _, proc, logf, _) in remaining.items():
        try: proc.send_signal(signal.SIGKILL); logf.close()
        except Exception: pass
    raise
print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 5. Copy results back to Drive.
import shutil, os
DECAYS = [0.02, 0.05, 0.10, 0.20]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for d in DECAYS:
    tag = f'dec{int(d*100):03d}'
    src = f'reports/phase34_saighi_{tag}_seed1'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_saighi_{tag}_seed1', dirs_exist_ok=True)
shutil.copytree('reports/phase34_saighi_decay_sweep_colab',
                f'{dst}/phase34_saighi_decay_sweep_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'saighi_dec'

In [ ]:
# 6. Summary: per-decay-value, show C-A deltas + A_max trajectory.
import json
from pathlib import Path

DECAYS = [0.02, 0.05, 0.10, 0.20]
# Include n=10 result's seed 1 (decay=0.0) as the reference point
print(f'{"decay":>6} {"top1_A":>7} {"top1_C":>7} {"Δtop1":>8} {"R10_A":>7} {"R10_C":>7} {"ΔR10":>8} {"cap_A":>7} {"cap_C":>7} {"Δcap":>8} {"A_max":>7} {"A_nz":>5}')
for d in DECAYS:
    tag = f'dec{int(d*100):03d}'
    p = Path(f'reports/phase34_saighi_{tag}_seed1/phase34_results.json')
    if not p.exists():
        print(f'{d:>6.2f} MISSING')
        continue
    data = json.loads(p.read_text())
    A = data['results']['baseline_static'][-1]
    C = data['results']['phase3_phase4'][-1]
    dd = C.get('death_diag', {}).get('2', {})
    print(f'{d:>6.2f} {A["top1"]:>7.4f} {C["top1"]:>7.4f} {C["top1"]-A["top1"]:>+8.4f} '
          f'{A["topk"]:>7.4f} {C["topk"]:>7.4f} {C["topk"]-A["topk"]:>+8.4f} '
          f'{A["cap_t_05"]:>7.4f} {C["cap_t_05"]:>7.4f} {C["cap_t_05"]-A["cap_t_05"]:>+8.4f} '
          f'{dd.get("inhibition_max", 0):>7.3f} {dd.get("inhibition_nonzero", 0):>5}')

print()
print('For reference, seed 1 at decay=0.0 (from n=10 sweep):')
print(f'  Δtop1=+0.0529, ΔR@10=+0.0264, Δcap_t05=+0.0176, A_max=13.61, A_nz=3')
print('(That was the outlier-positive seed in the n=10. We expect decay>0 to TIGHTEN A_k,')
print(' likely making Δtop1 smaller in magnitude. If it goes more positive at higher decay,')
print(' the mechanism would benefit from forgiveness. If it goes negative, decay alone won\'t save A_k.)')